_**Setup & Imports**_

In [ ]:
# Core data analysis libraries
import numpy as np
import pandas as pd
from plotnine import *  # generally not a good thing to do to import everything from a package. However it's ok for visualization purposes in an analysis.
import os
import scipy
import warnings
from IPython.core.interactiveshell import InteractiveShell

InteractiveShell.ast_node_interactivity = "all"  # to make jupyter print all outputs, not just the last one
from IPython.core.display import HTML  # to pretty print pandas df and be able to copy them over (e.g. to ppt slides)


_**Load the Data**_

In [ ]:
# Load datasets
# Each dataset represents a different aspect of movie data

expert_df = pd.read_csv("../../Metacritic dataset/ExpertReviews.csv")

user_df = pd.read_csv(
    "../../Metacritic dataset/UserReviews.csv",
    low_memory=False  # avoids dtype inference warnings
)

meta_df = pd.read_csv("../../Metacritic dataset/metaClean43Brightspace.csv")

sales_df = pd.read_csv("../../Metacritic dataset/sales.csv")


**Print the dataset row and columns**

In [ ]:
print("There are {} rows and {} columns in the ExpertReviews.csv ".format(expert_df.shape[0], expert_df.shape[1]))
print("There are {} rows and {} columns in the UserReviews.csv ".format(user_df.shape[0], user_df.shape[1]))
print("There are {} rows and {} columns in the metaClean43Brightspace.csv ".format(meta_df.shape[0], meta_df.shape[1]))
print("There are {} rows and {} columns in the sales.csv ".format(sales_df.shape[0], sales_df.shape[1]))

Print column names

In [ ]:
print("Column names in ExpertReviews.csv:", list(expert_df.columns))
print("Columns names in UserReviews.csv ", list(user_df.columns))
print("Columns names in metaClean43Brightspace.csv ", list(meta_df.columns))
print("Columns names in sales.csv ", list(sales_df.columns))

We see the names of the columns in each dataset, but we can see that names of some columns are not good, so we will change them to better names.

In [ ]:
expert_df = expert_df.rename(columns={
    'idvscore': 'individual_score',
    'dateP': 'publish_date',
    'Rev': 'review_text'
})

user_df = user_df.rename(columns={
    'idvscore': 'individual_score',
    'dateP': 'publish_date',
    'Rev': 'review_text',
    'thumbsUp': 'thumbs_up',
    'thumbsTot': 'thumbs_total'
})

meta_df = meta_df.rename(columns={
    'RelDate': 'release_date',
    'metascore': 'meta_score',
    'userscore': 'user_score'
})

sales_df = sales_df.rename(columns={
    'international_box_office': 'intl_box_office',
    'domestic_box_office': 'dom_box_office',
    'worldwide_box_office': 'global_box_office',
    'avg run per theatre': 'avg_run_per_theatre',
    'Unnamed: 8': 'opening_weekend_revenue'
})



Let's how the names of the columns has changed

In [ ]:
print("Column names in ExpertReviews.csv:", list(expert_df.columns))
print("Columns names in UserReviews.csv ", list(user_df.columns))
print("Columns names in metaClean43Brightspace.csv ", list(meta_df.columns))
print("Columns names in sales.csv ", list(sales_df.columns))

Let's check the percentage of the missing values of each column

In [ ]:
def missing_summary(df, name):
    summary = (
        df.isna()
        .sum()
        .to_frame(name='Missing Values')
        .assign(Percentage=lambda x: (x['Missing Values'] / len(df) * 100).round(2))
    )
    summary = summary[summary['Missing Values'] > 0]

    print(f"\n{name} — Missing Values Summary")
    print(summary if not summary.empty else "No missing values 🎉")




In [ ]:
missing_summary(expert_df, "ExpertReviews")
missing_summary(user_df, "UserReviews")
missing_summary(meta_df, "Meta")
missing_summary(sales_df, "Sales")

In [ ]:
# Drop fully empty columns
sales_df = sales_df.drop(columns=[
    'opening_weekend_revenue'
])

print("Columns names in sales.csv ", list(sales_df.columns))

Let's check on the type of the columns and correct where it's needed

In [ ]:
print("ExpertReviews dtypes:\n", expert_df.dtypes, "\n")
print("UserReviews dtypes:\n", user_df.dtypes, "\n")
print("Meta dtypes:\n", meta_df.dtypes, "\n")
print("Sales dtypes:\n", sales_df.dtypes)


We fix here the datetypes to be correct

In [ ]:
expert_df['publish_date'] = pd.to_datetime(
    expert_df['publish_date'],
    errors='coerce'
)


Fix the release date and the year in the sales table

In [ ]:
# 1. Clean release_date text
# 1. Clean release_date text (KEEP THIS)
sales_df['release_date_clean'] = (
    sales_df['release_date']
    .astype(str)
    .str.replace(r'\(.*?\)', '', regex=True)
    .str.replace(r'\b(st|nd|rd|th)\b', '', regex=True)
    .str.strip()
)

# 2. Create fixed release_date ONLY when month + day exist (FIXED)
month_regex = r'^(January|February|March|April|May|June|July|August|September|October|November|December)\s+\d{1,2}$'

mask = sales_df['release_date_clean'].str.match(month_regex)

sales_df['release_date_fixed'] = pd.NaT
sales_df.loc[mask, 'release_date_fixed'] = pd.to_datetime(
    sales_df.loc[mask, 'release_date_clean'] + ' ' + sales_df.loc[mask, 'year'].astype(str),
    format='%B %d %Y',
    errors='coerce'
)



In [ ]:
sales_df['release_date_fixed'].notna().mean() * 100


We convert publish_date to datetime

In [ ]:
expert_df['publish_date'] = pd.to_datetime(
    expert_df['publish_date'], errors='coerce'
)

We convert the following:
individual_score to numeric
thumbs_up and thumbs_total to Int64
publish_date to datetime


In [ ]:
user_df['individual_score'] = pd.to_numeric(
    user_df['individual_score'], errors='coerce'
)

user_df['thumbs_up'] = pd.to_numeric(
    user_df['thumbs_up'], errors='coerce'
)

user_df['thumbs_total'] = pd.to_numeric(
    user_df['thumbs_total'], errors='coerce'
)

user_df['publish_date'] = pd.to_datetime(
    user_df['publish_date'], errors='coerce'
)

user_df['thumbs_up'] = user_df['thumbs_up'].astype('Int64')
user_df['thumbs_total'] = user_df['thumbs_total'].astype('Int64')



We convert the following:
release_date to datetime
rating, genre, studio to category

In [ ]:
meta_df['release_date'] = pd.to_datetime(
    meta_df['release_date'], errors='coerce'
)

categorical_cols = ['rating', 'genre', 'studio']
for col in categorical_cols:
    meta_df[col] = meta_df[col].astype('category')


We convert the following:
release_date to datetime
genre, creative_type to category

In [ ]:
sales_df['genre'] = sales_df['genre'].astype('category')
sales_df['creative_type'] = sales_df['creative_type'].astype('category')


Check on the datatypes after making the converts

In [ ]:
print("ExpertReviews dtypes:\n", expert_df.dtypes, "\n")
print("UserReviews dtypes:\n", user_df.dtypes, "\n")
print("Meta dtypes:\n", meta_df.dtypes, "\n")
print("Sales dtypes:\n", sales_df.dtypes)
